Подключаем Google Disk для доступа к файлам:

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")

Набор папок, которые являются временными при дообучении модели. Ячейка снизу способна удалить их, если это потребуется:

In [ ]:
# ============================ To remove unnecessary folders/files ============================
import os
import shutil

if os.path.exists("audio_files"):
    shutil.rmtree("audio_files")
if os.path.exists("rttm_files"):
    shutil.rmtree("rttm_files")
if os.path.exists("uem_files"):
    shutil.rmtree("uem_files")
if os.path.exists("lightning_logs"):
    shutil.rmtree("lightning_logs")
if os.path.exists("/content/drive/MyDrive/lightning_logs"):
    shutil.rmtree("/content/drive/MyDrive/lightning_logs")
if os.path.exists("test.lst"):
    os.remove("test.lst")
if os.path.exists("train.lst"):
    os.remove("train.lst")
if os.path.exists("validation.lst"):
    os.remove("validation.lst")
if os.path.exists("database.yml"):
    os.remove("database.yml")

Установка необходимых зависимостей:

In [ ]:
!pip install pyannote.audio pyannote.database

Чтобы не было конфликтов библиотек pytorch-lightning и lightning, нужно удалить их обе, а потом скачать только lightning:

In [ ]:
!pip uninstall -y pytorch-lightning lightning
!pip install lightning

Обучение с помощью Lightning. Ключевые особенности обучения:
1. Использование EarlyStopping, если улучшения не было какое-то количество эпох
2. 35 эпох для обучения
3. Выполнение разогревочного обучения перед основным с заморозкой слоев
4. Обучение использует ReduceLROnPlateau для корректироваки Learning Rate у Adam во время обучения
5. Используется оптимизатор Adam для подбора шага для каждого параметра. Он использует learning rate, который задает общий масштаб шага.
6. Валидация происходит два раза за эпоху: посередине и в конце

In [ ]:
import os
import torch
import lightning as L
import multiprocessing
from google.colab import userdata
from pyannote.audio import Model
from pyannote.audio.tasks import SpeakerDiarization
from pyannote.database import registry, FileFinder
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.serialization import add_safe_globals
from pyannote.audio.core.task import Problem, Resolution, Specifications

add_safe_globals([Problem, Resolution, Specifications])

# ========= Start Config
CPU_COUNT = multiprocessing.cpu_count()
ACCELERATOR = "gpu" if torch.cuda.is_available() else "cpu"
MONITOR_METRIC = "DiarizationErrorRate"
BATCH_SIZE = 16
WARMUP_EPOCHS = 3
MAX_EPOCHS = 35
NUM_WORKERS = min(CPU_COUNT, 2) if ACCELERATOR == "gpu" else CPU_COUNT
DURATION = 5.0
LEARNING_RATE = 5e-5
EXP_DIR = "/content/drive/MyDrive/pyannote_finetuning_3.1"
LOG_NAME = "ami_segmentation_v1"
DATABASE_CONFIG_FILE = "/content/drive/MyDrive/AMI-diarization-setup/pyannote/database.yml"
# ========= End Config

print("Load database...")
registry.load_database(DATABASE_CONFIG_FILE)
protocol = registry.get_protocol("AMI.SpeakerDiarization.word_and_vocalsounds", preprocessors={"audio": FileFinder()})

print("Download model...")
model = Model.from_pretrained(
    "pyannote/segmentation-3.1",
    use_auth_token=userdata.get('HF_TOKEN')
)

print("Create task...")
task = SpeakerDiarization(
    protocol,
    duration=DURATION,
    max_speakers_per_chunk=4,
    max_speakers_per_frame=3,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,

)
model.task = task
print("Model prepare data...")
model.prepare_data()

def configure_optimizers():
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=0.001
    )
    scheduler = ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=4
    )
    return {
        "optimizer": optimizer,
        "lr_scheduler": {
            "scheduler": scheduler,
            "monitor": MONITOR_METRIC,
            "interval": "epoch",
            "frequency": 1
        }
    }

model.configure_optimizers = configure_optimizers
print("Create logger...")
logger = CSVLogger(save_dir=EXP_DIR, name=LOG_NAME, version="fixed_version")

print("Create checkpoint callback...")
checkpoint_callback = ModelCheckpoint(
    monitor=MONITOR_METRIC,
    mode="min",
    save_top_k=3,
    save_last=True,
    filename="best-{epoch:02d}-{step}",
)

print("Create early stop callback...")
early_stop_callback = EarlyStopping(
    monitor=MONITOR_METRIC,
    mode="min",
    patience=12,
    verbose=True,
)

ckpt_path = os.path.join(EXP_DIR, LOG_NAME, "fixed_version", "checkpoints", "last.ckpt")
if os.path.exists(ckpt_path):
    print(f"A checkpoint has been found! Resuming training with: {ckpt_path}")
    resume_path = ckpt_path
else:
    print("No checkpoints were found. We are starting a new training")
    resume_path = None

precision = "16-mixed" if ACCELERATOR == "gpu" else "32-true"

if resume_path is not None:
    print("--- Stage 1: SKIPPED ---")
    for param in model.parameters():
        param.requires_grad = True
else:
    print("--- Stage 1: Warmup ---")
    for name, param in model.named_parameters():
        if "classifier" in name or "activation" in name:
            param.requires_grad = True
        else:
            param.requires_grad = False
    warmup_trainer = L.Trainer(
        max_epochs=WARMUP_EPOCHS,
        accelerator=ACCELERATOR,
        devices=1,
        gradient_clip_val=0.5,
        precision=precision,
        logger=logger,
        enable_checkpointing=False
    )
    print("Warmup fit...")
    warmup_trainer.fit(model)
    for param in model.parameters():
        param.requires_grad = True

print("--- Stage 2: Final fine-tuning ---")
trainer = L.Trainer(
    default_root_dir=EXP_DIR,
    max_epochs=MAX_EPOCHS,
    accelerator=ACCELERATOR,
    devices=1,
    gradient_clip_val=0.5,
    precision=precision,
    callbacks=[checkpoint_callback, early_stop_callback],
    logger=logger,
    log_every_n_steps=5,
    val_check_interval=0.5
)

print("Trainer fit...")
trainer.fit(model, ckpt_path=resume_path)

print("Save pretrained model...")
final_save_path = os.path.join(EXP_DIR, "ami-segmentation-final")